# Adaptive Retrieval for RAG — From First Principles

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/adaptive_retrieval.ipynb)

Adaptive retrieval asks a simple systems question: **should every query be retrieved in the same way?** In basic RAG, the answer is usually yes: fixed top-k, fixed threshold, fixed prompt, fixed generation path. That simplicity is useful, but it ignores a real property of user questions: different questions need different retrieval behaviors.

A short factual query such as *"What year did HelioGrid launch phase-two storage?"* wants a narrow and precise retrieval window. A broad analytical query such as *"How do grid batteries and demand response complement each other?"* benefits from a wider evidence set. Comparative questions often need parallel evidence packets rather than one ranked list. Adaptive retrieval is the layer that turns those differences into explicit routing logic.

This notebook builds that logic from scratch with native Python. No chain framework decides the route for us. We will classify the query, pick a strategy, adapt the retrieval budget, and only then generate an answer.

## Why a Single Retrieval Policy Breaks Down

A fixed policy is attractive because it reduces engineering choices. But it also hides three separate assumptions:

1. Every query has the same ambiguity level.
2. Every query needs the same amount of evidence.
3. Every query benefits from the same precision/recall tradeoff.

Those assumptions are false in practice. Good retrieval systems are really **decision systems**. They decide whether to go narrow or broad, whether to decompose, and whether to retry when the first pass is weak.

## A Minimal Adaptive Retrieval Architecture

```
question
  -> classify intent
  -> select retrieval strategy
  -> retrieve with strategy-specific parameters
  -> inspect confidence
  -> widen / reformulate / decompose if needed
  -> generate grounded answer
```

The important design principle is that adaptation happens **before** generation and sometimes **instead of** generation. If retrieval is weak, the system should change the search behavior rather than confidently writing a weak answer.

## Environment Setup

All notebooks in this overhaul use the same minimal native stack:

- **Qwen/Qwen3-8B** for generation
- **BAAI/bge-base-en-v1.5** for embeddings
- **FAISS** for similarity search
- **Google Drive** for persistent Colab caching

The goal is to keep the pipeline transparent. Every major component is visible as plain Python functions instead of framework abstractions.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy
print("Installed core native RAG dependencies.")

In [ ]:
import time
import math
import json
import re
from collections import defaultdict, Counter

import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print("Loaded", MODEL_NAME)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print("Loaded embedding model:", embed_model.__class__.__name__)

In [ ]:
def generate(prompt, max_new_tokens=220, temperature=0.2):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        top_k=20,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    answer_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

print("Generation helper ready.")

## A Synthetic Knowledge Base with Multiple Query Types

To make adaptation visible, we need a corpus that supports different kinds of questions: factual, comparative, analytical, and procedural. The knowledge base below mixes company facts, energy-system explanations, and policy notes so the route choice matters.

In [ ]:
knowledge_base = [
    {"id": "d1", "topic": "solar", "text": "HelioGrid launched its first utility-scale solar farm in 2018 and expanded with phase-two battery storage in 2022. The company says the storage layer reduced curtailment during midday oversupply."},
    {"id": "d2", "topic": "solar", "text": "Solar generation is highly predictable at seasonal scale but volatile at cloud-scale. Operators often combine short-duration batteries with inverter controls to smooth the output profile."},
    {"id": "d3", "topic": "storage", "text": "Grid batteries respond in milliseconds and are effective for frequency stabilization. They are weaker at long-duration seasonal shifting unless paired with larger storage chemistries or demand flexibility."},
    {"id": "d4", "topic": "demand", "text": "Demand response reduces consumption during peak periods by shifting industrial loads, smart appliances, and electric vehicle charging. It works as a virtual flexibility resource."},
    {"id": "d5", "topic": "wind", "text": "Northwind Energy operates offshore turbines whose output tends to peak at night. This complements daytime solar generation and reduces net load volatility when both are coordinated."},
    {"id": "d6", "topic": "policy", "text": "The 2025 Flex Grid Act introduced performance payments for storage, demand response, and dispatchable clean generation. It favored resources that could provide grid services during evening peaks."},
    {"id": "d7", "topic": "comparison", "text": "Batteries are best for fast response and short shifts. Hydrogen storage is slower but potentially useful for long-duration balancing. Their economic value depends on market structure and utilization frequency."},
    {"id": "d8", "topic": "comparison", "text": "Pumped hydro has high upfront capital cost but long asset lifetime. Lithium-ion is modular and fast to deploy but degrades more quickly under heavy cycling."},
    {"id": "d9", "topic": "analysis", "text": "A resilient grid usually needs portfolio effects. Solar, wind, storage, transmission, and flexible demand each reduce a different failure mode, so diversity matters more than any single technology."},
    {"id": "d10", "topic": "analysis", "text": "Transmission expansion lets geographically separated renewable resources balance one another. It also lowers the total amount of local storage needed in some system designs."},
    {"id": "d11", "topic": "procedure", "text": "To compare two resources, analysts often examine response speed, energy duration, capital cost, operating cost, and transmission dependence. A simple ranked list usually hides these tradeoffs."},
    {"id": "d12", "topic": "procedure", "text": "When a query combines policy and engineering, retrieval usually benefits from broader recall because the relevant evidence is distributed across regulatory and technical passages."},
]

print("Documents:", len(knowledge_base))
print("Topics:", sorted({doc["topic"] for doc in knowledge_base}))

In [ ]:
texts = [doc["text"] for doc in knowledge_base]
doc_embeddings = embed_model.encode(texts, normalize_embeddings=True)
doc_embeddings = np.asarray(doc_embeddings, dtype="float32")

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("Embedding matrix shape:", doc_embeddings.shape)
print("FAISS vectors stored:", index.ntotal)

## A Fixed Baseline

We start with the deliberately naive baseline: always retrieve the top 3 chunks. This gives us a stable reference point for later comparisons.

In [ ]:
def baseline_retrieve(question, k=3):
    query_vec = embed_model.encode([question], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype="float32")
    scores, indices = index.search(query_vec, k)
    return [
        {
            "rank": i + 1,
            "score": float(score),
            "doc_id": knowledge_base[idx]["id"],
            "topic": knowledge_base[idx]["topic"],
            "text": knowledge_base[idx]["text"],
        }
        for i, (score, idx) in enumerate(zip(scores[0], indices[0]))
    ]

sample_query = "How do batteries and demand response work together during evening peaks?"
for item in baseline_retrieve(sample_query):
    print(item["rank"], item["topic"], round(item["score"], 3), item["doc_id"])

## Query Classification as a Routing Problem

Adaptive retrieval starts with classification. We do not need a giant orchestration framework for this. We need a small, inspectable function that decides what kind of question we are facing.

The classes below are intentionally practical:

- **factual**: narrow answer, usually one or two passages
- **analytical**: broad synthesis across several passages
- **comparative**: parallel evidence about multiple entities or options
- **procedural**: method-oriented or step-by-step question

In [ ]:
def classify_query(question):
    q = question.lower()
    if any(token in q for token in ["compare", "versus", "vs", "difference"]):
        return "comparative"
    if any(token in q for token in ["how do", "why do", "tradeoff", "interact", "work together"]):
        return "analytical"
    if any(token in q for token in ["how should", "what steps", "method", "procedure"]):
        return "procedural"
    return "factual"

examples = [
    "What year did HelioGrid expand storage?",
    "How do batteries and demand response work together?",
    "Compare hydrogen storage versus batteries.",
    "What steps should an analyst use to compare two grid resources?",
]

for q in examples:
    print(f"{classify_query(q):>11} | {q}")

## Strategy Routing

Once we know the class, we can choose retrieval settings intentionally rather than implicitly. The router below expresses a simple design idea: analytical and policy-heavy questions need more recall; factual ones need more precision.

In [ ]:
STRATEGIES = {
    "factual": {"k": 2, "min_score": 0.48, "mode": "direct"},
    "analytical": {"k": 5, "min_score": 0.36, "mode": "broad"},
    "comparative": {"k": 4, "min_score": 0.34, "mode": "pairwise"},
    "procedural": {"k": 4, "min_score": 0.38, "mode": "stepwise"},
}

for name, cfg in STRATEGIES.items():
    print(name, "->", cfg)

## Dynamic K Selection

Instead of fixing k once for the whole system, we can let the query type and wording influence it. This is not magic; it is just explicit policy. If a question asks for comparison or system interaction, we widen the retrieval budget.

In [ ]:
def dynamic_k(question, query_type):
    base = STRATEGIES[query_type]["k"]
    q = question.lower()
    if "and" in q or "compare" in q or "versus" in q:
        base += 1
    if "policy" in q and query_type != "factual":
        base += 1
    return min(base, len(knowledge_base))

for q in examples:
    query_type = classify_query(q)
    print(query_type, "->", dynamic_k(q, query_type), "|", q)

## Confidence-Based Adaptation

A retrieval policy should not stop after its first attempt if the score profile is weak. We can inspect the top scores and decide whether to widen the search budget. This is the simplest possible corrective loop.

In [ ]:
def retrieve_with_strategy(question):
    query_type = classify_query(question)
    k = dynamic_k(question, query_type)
    query_vec = embed_model.encode([question], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype="float32")
    scores, indices = index.search(query_vec, k)

    docs = []
    for score, idx in zip(scores[0], indices[0]):
        docs.append({
            "score": float(score),
            "doc": knowledge_base[int(idx)],
        })

    min_score = STRATEGIES[query_type]["min_score"]
    filtered = [item for item in docs if item["score"] >= min_score]
    widened = False

    if len(filtered) < 2 and k < len(knowledge_base):
        widened = True
        scores, indices = index.search(query_vec, min(k + 2, len(knowledge_base)))
        filtered = []
        for score, idx in zip(scores[0], indices[0]):
            if float(score) >= min_score - 0.03:
                filtered.append({"score": float(score), "doc": knowledge_base[int(idx)]})

    return {
        "query_type": query_type,
        "k": k,
        "widened": widened,
        "results": filtered
    }

result = retrieve_with_strategy("Compare batteries versus hydrogen storage for peak support.")
print("Type:", result["query_type"])
print("Original k:", result["k"], "| Widened:", result["widened"])
for item in result["results"]:
    print(round(item["score"], 3), "|", item["doc"]["topic"], "|", item["doc"]["id"])

## Comparative Queries Need Structured Evidence

Comparative questions often fail when retrieval returns one dominant concept and ignores the rival concept. Below we split the question into two facets and retrieve evidence for each side before merging.

In [ ]:
def split_comparative_query(question):
    q = question.lower()
    if " versus " in q:
        left, right = q.split(" versus ", 1)
        return [left.strip(), right.strip()]
    if " vs " in q:
        left, right = q.split(" vs ", 1)
        return [left.strip(), right.strip()]
    return [question]

parts = split_comparative_query("Compare batteries versus hydrogen storage for peak support.")
print(parts)

In [ ]:
def comparative_retrieve(question):
    parts = split_comparative_query(question)
    merged = []
    seen = set()
    for part in parts:
        for item in baseline_retrieve(part, k=3):
            if item["doc_id"] not in seen:
                seen.add(item["doc_id"])
                merged.append(item)
    return merged

for item in comparative_retrieve("Compare batteries versus hydrogen storage for peak support."):
    print(item["doc_id"], item["topic"], round(item["score"], 3))

## Building the Adaptive Orchestrator

The orchestrator ties together three ideas:

1. classify the question,
2. route to the right retrieval policy,
3. build a context packet whose structure matches the query type.

In [ ]:
def build_context(question):
    query_type = classify_query(question)
    if query_type == "comparative":
        rows = comparative_retrieve(question)
        context = "\n\n".join([f"[{row['doc_id']}] {row['text']}" for row in rows[:4]])
        return query_type, context, rows[:4]

    payload = retrieve_with_strategy(question)
    context = "\n\n".join([f"[{item['doc']['id']}] {item['doc']['text']}" for item in payload["results"]])
    return payload["query_type"], context, payload["results"]

query_type, context, evidence = build_context("How do batteries and demand response work together?")
print("Type:", query_type)
print("Evidence items:", len(evidence))
print(context[:500])

In [ ]:
def adaptive_answer(question):
    query_type, context, evidence = build_context(question)
    prompt = f"""
You are answering a {query_type} question.
Use only the evidence below. If evidence is weak or incomplete, say that directly.

Evidence:
{context}

Question: {question}

Answer:
""".strip()
    answer = generate(prompt, max_new_tokens=180)
    return {
        "query_type": query_type,
        "evidence_count": len(evidence),
        "answer": answer
    }

demo = adaptive_answer("How do batteries and demand response work together during evening peaks?")
print(json.dumps(demo, indent=2))

## Baseline vs Adaptive

Adaptation is only useful if it changes the evidence packet in a meaningful way. The next cell compares the fixed baseline against the adaptive policy on a mix of query types.

In [ ]:
evaluation_queries = [
    "What year did HelioGrid expand storage?",
    "How do batteries and demand response work together during evening peaks?",
    "Compare batteries versus hydrogen storage for peak support.",
    "What steps should an analyst use to compare two grid resources?",
    "How does policy interact with engineering flexibility on the grid?",
]

rows = []
for q in evaluation_queries:
    baseline_docs = [item["doc_id"] for item in baseline_retrieve(q, k=3)]
    adaptive_payload = build_context(q)[2]
    adaptive_docs = [item["doc_id"] if "doc_id" in item else item["doc"]["id"] for item in adaptive_payload]
    rows.append({
        "question": q,
        "type": classify_query(q),
        "baseline_docs": baseline_docs,
        "adaptive_docs": adaptive_docs[:5],
    })

for row in rows:
    print("\nQUESTION:", row["question"])
    print("TYPE:", row["type"])
    print("BASELINE:", row["baseline_docs"])
    print("ADAPTIVE:", row["adaptive_docs"])

## A Traceable Adaptive Pipeline Class

Production systems need inspectable traces. A good adaptive system should be able to tell you **why** it chose a route and whether it widened the search because confidence was low.

In [ ]:
class AdaptiveRAGPipeline:
    def run(self, question):
        query_type = classify_query(question)
        trace = {"question": question, "query_type": query_type}

        if query_type == "comparative":
            evidence = comparative_retrieve(question)
            trace["mode"] = "pairwise"
            trace["widened"] = False
            context = "\n\n".join([f"[{item['doc_id']}] {item['text']}" for item in evidence[:4]])
        else:
            retrieval = retrieve_with_strategy(question)
            evidence = retrieval["results"]
            trace["mode"] = STRATEGIES[query_type]["mode"]
            trace["widened"] = retrieval["widened"]
            context = "\n\n".join([f"[{item['doc']['id']}] {item['doc']['text']}" for item in evidence])

        trace["evidence_count"] = len(evidence)
        trace["answer"] = generate(
            f"Answer using only the evidence below.\n\nEvidence:\n{context}\n\nQuestion: {question}\n\nAnswer:",
            max_new_tokens=160,
        )
        return trace

pipeline = AdaptiveRAGPipeline()
trace = pipeline.run("How does policy interact with engineering flexibility on the grid?")
print(json.dumps(trace, indent=2))

## Latency and Complexity

Adaptive retrieval is not free. Each layer of routing, splitting, or widening adds code paths and often extra retrieval calls. The payoff is better control over evidence quality. The cost is more logic, more tuning, and more monitoring.

That is why adaptive retrieval should be framed as **policy engineering**. The job is not to add complexity for its own sake. The job is to create a small set of route choices that solve clearly different query shapes.

In [ ]:
start = time.time()
for q in evaluation_queries:
    _ = pipeline.run(q)
elapsed = time.time() - start
print(f"Processed {len(evaluation_queries)} adaptive queries in {elapsed:.2f}s")
print(f"Average per query: {elapsed / len(evaluation_queries):.2f}s")

## When to Use Adaptive Retrieval

Adaptive retrieval is most useful when:

- your workload has clearly different query families,
- retrieval mistakes are costly,
- and you need explicit control over precision vs recall.

It is less useful when the corpus is tiny, the questions are homogeneous, or a simple hybrid retriever already performs well. In those cases, a fixed policy is easier to maintain.

In [ ]:
decision_matrix = [
    ["Homogeneous FAQs", "Low", "Fixed retrieval is enough"],
    ["Mixed factual + analytical support", "High", "Adaptive retrieval helps"],
    ["Heavy comparison workflows", "High", "Adaptive retrieval strongly helps"],
    ["Tiny personal notes corpus", "Low", "Adaptation is overkill"],
]

for row in decision_matrix:
    print(" | ".join(row))

## Key Takeaways

Adaptive retrieval is not one algorithm. It is a design stance: **the retrieval plan should respond to the question**. Once you see RAG this way, techniques like query decomposition, corrective retrieval, and agentic tool use become natural extensions of the same idea.

The main lesson is simple: retrieval quality improves when the system is allowed to behave differently for different information needs.